Dirichlet Lame Indirect Method
=============================

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve.webgui import Draw
from ngsolve.bem import *
from ngsolve import Projector, Preconditioner
from ngsolve.krylovspace import CG

Define the geometry $\Omega \subset \mathbb R^3$ and create a mesh:

In [ ]:
sp = Glue ( Sphere( (0,0,0), 1).faces)
mesh = Mesh( OCCGeometry(sp).GenerateMesh(maxh=1)).Curve(3)

Create test and trial function finite element spaces for $H^{-\frac12}(\Gamma)$ according to the given mesh:  

In [ ]:
fesL2 = VectorValued(SurfaceL2(mesh, order=3, dual_mapping=False))
# fesL2 = VectorH1(mesh, order=3)
print (fesL2.ndof)
u,v = fesL2.TnT()

Define Dirichlet data $u_0$ and compute the right hand side vector:

In [ ]:
p0 = CF( (2,2,2) )
X = CF( (x,y,z) )

E, nu = 210, 0.2
alpha = (1+nu)/((1-nu)*2*E)
norm = Norm(X-p0)
lapkernel = alpha/(4*pi) * 1/norm

u0 = (3-4*nu) * lapkernel * CF( (1,0,0) ) \
   + lapkernel/norm**2 * (X-p0) * (X-p0)[0] 
u0 *= 1000
rhs = LinearForm (u0*v.Trace()*ds(bonus_intorder=3)).Assemble()

Draw (u0, mesh)

In [ ]:
j = GridFunction(fesL2)
pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()

with TaskManager():
    V = LameSL(u*ds,E,nu) *v*ds
    CG(mat = V.mat, pre=pre, rhs = rhs.vec, sol=j.vec, tol=1e-8, maxsteps=100, initialize=False, printrates=True)


In [ ]:
Draw (j, order=3);

In [ ]:
vismesh = (WorkPlane().RectangleC(4,4).Face()*Sphere((0,0,0),1)).GenerateMesh(maxh=0.1).Curve(4)
sol = GridFunction(VectorH1(vismesh,order=3))

SL = LameSL(u*ds(bonus_intorder=4), E, nu)(j)
# SL = (LameSL(u*ds(bonus_intorder=4), E, nu) * v*ds) . GetPotential(j)

sol.Set (SL, definedon=vismesh.Boundaries(".*"))
Draw (sol, vismesh, order=3);
Draw (u0, vismesh, order=3);
Draw (sol-u0, vismesh, order=3);